# Week 4 Lab: ViT, CLIP, VLMs, and a PathVQA-Style Capstone

This notebook is the CPU-friendly core route for Week 4 of **Deep Learning in 100 Hours**. It uses generated images and toy text only. It cannot support medical conclusions, diagnosis, or claims about real PathVQA performance.

| Day | Main idea | Evidence to keep |
|---|---|---|
| 16 | Images as ViT patch tokens | Patch/CLS shapes and tiny overfit result |
| 17 | CLIP-style contrastive alignment | Aligned loss and retrieval beat permutation |
| 18 | Lightweight image-question fusion | Full model beats unimodal and shuffle controls |
| 19 | PathVQA protocol and limitations | Stratified metrics, data card, failure records |
| 20 | Reproducible capstone package | Result table, manifest, and oral defense |

**Core Light:** every cell here. **Optional Heavy:** real PathVQA access, pretrained vision/language models, GPU fine-tuning, and a medically meaningful error review. External resources must be checked for source, license, de-identification, and split policy before use.

In [ ]:
from collections import Counter, defaultdict
import json

import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F

from dl100h.cnn import make_stripe_images
from dl100h.pathvqa import TOY_PATHVQA, exact_match, normalize_answer, token_f1
from dl100h.utils import count_parameters, seed_everything
from dl100h.vision import TinyCLIP, TinyViT, clip_contrastive_loss
from dl100h.vlm import TinyVLMClassifier

seed_everything(42)
device = torch.device('cpu')
print({'torch': torch.__version__, 'device': str(device)})

# Day 16 — Vision Transformers: images as token sequences

A ViT divides an image into non-overlapping patches and projects each patch to a token. For image size $H\times W$ and square patch size $P$, the number of patches is

$$N=\frac{H}{P}\frac{W}{P}.$$

A classification token is prepended, position embeddings are added, and a Transformer encoder mixes information across patches. Smaller patches increase sequence length and attention cost; larger patches reduce detail.

In [ ]:
seed_everything(16)
stripe_images_gray, stripe_labels = make_stripe_images(n=24, image_size=32, seed=16)
stripe_images = stripe_images_gray.repeat(1, 3, 1, 1)
vit = TinyViT(
    image_size=32,
    patch_size=8,
    in_channels=3,
    dim=16,
    depth=1,
    n_heads=4,
    n_classes=2,
).to(device)

with torch.inference_mode():
    patch_tokens = vit.patch(stripe_images.to(device))
    all_tokens = vit.forward_features(stripe_images.to(device))
    initial_logits = vit(stripe_images.to(device))

assert patch_tokens.shape == (24, 16, 16)
assert all_tokens.shape == (24, 17, 16)
assert initial_logits.shape == (24, 2)
{
    'image_shape': tuple(stripe_images.shape),
    'patch_token_shape': tuple(patch_tokens.shape),
    'tokens_with_cls_shape': tuple(all_tokens.shape),
    'parameters': count_parameters(vit),
}

In [ ]:
vit_optimizer = torch.optim.Adam(vit.parameters(), lr=0.02)
vit_losses = []
vit.train()
for step in range(80):
    vit_logits = vit(stripe_images.to(device))
    vit_loss = F.cross_entropy(vit_logits, stripe_labels.to(device))
    vit_optimizer.zero_grad(set_to_none=True)
    vit_loss.backward()
    vit_optimizer.step()
    vit_losses.append(vit_loss.item())

vit.eval()
with torch.inference_mode():
    vit_predictions = vit(stripe_images.to(device)).argmax(dim=-1).cpu()
vit_accuracy = (vit_predictions == stripe_labels).float().mean().item()
assert vit_accuracy >= 0.95
{'initial_loss': vit_losses[0], 'final_loss': vit_losses[-1], 'tiny_batch_accuracy': vit_accuracy}

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(9, 3.2))
axes[0].plot(vit_losses)
axes[0].set(xlabel='optimization step', ylabel='cross-entropy', title='Tiny ViT overfit curve')
axes[1].imshow(stripe_images[0].permute(1, 2, 0).clamp(0, 1))
axes[1].set_title(f'label={stripe_labels[0].item()}, prediction={vit_predictions[0].item()}')
axes[1].axis('off')
figure.tight_layout()
plt.show()

## Day 16 practice and acceptance

1. Predict token counts for patch sizes 4, 8, and 16 before changing the model.
2. Compare parameter count and runtime with the small CNN from Week 1.
3. Remove position embeddings and describe what spatial information is lost.
4. Compare CLS-token pooling with mean patch pooling.

**Accept the day when:** you can derive the patch count, trace $(B,N,C)$ including CLS, explain the patch-size trade-off, and overfit the tiny synthetic image batch.

# Day 17 — CLIP-style contrastive image-text alignment

CLIP-style training maps images and texts to a shared normalized embedding space. For a batch of paired features, the similarity matrix contains all image-text comparisons. The symmetric objective applies cross-entropy from image to text and from text to image.

In-batch negatives can include false negatives when different captions describe the same concept. Retrieval success can also come from shortcuts, so inspect data and compare against shuffled pairs.

In [ ]:
torch.manual_seed(17)
latent_pairs = torch.eye(8)
image_features = latent_pairs + 0.01 * torch.randn_like(latent_pairs)
text_features = latent_pairs + 0.01 * torch.randn_like(latent_pairs)
permuted_text_features = text_features.roll(shifts=1, dims=0)

aligned_clip_loss = clip_contrastive_loss(image_features, text_features)
permuted_clip_loss = clip_contrastive_loss(image_features, permuted_text_features)
similarities = F.normalize(image_features, dim=-1) @ F.normalize(text_features, dim=-1).T
image_to_text = similarities.argmax(dim=-1)
retrieval_at_one = (image_to_text == torch.arange(8)).float().mean().item()

assert aligned_clip_loss < permuted_clip_loss
assert retrieval_at_one >= 0.95
{
    'aligned_loss': aligned_clip_loss.item(),
    'permuted_loss': permuted_clip_loss.item(),
    'image_to_text_retrieval_at_1': retrieval_at_one,
}

In [ ]:
seed_everything(170)
tiny_clip = TinyCLIP(vocab_size=32, dim=16, image_size=32)
clip_images = torch.randn(4, 3, 32, 32)
clip_text_ids = torch.tensor([
    [1, 2, 3, 4],
    [5, 6, 7, 8],
    [9, 10, 11, 12],
    [13, 14, 15, 16],
])
tiny_clip_loss = tiny_clip(clip_images, clip_text_ids)
tiny_clip_loss.backward()

assert tiny_clip.image_encoder.head.weight.grad is not None
assert tiny_clip.text_projection.weight.grad is not None
{
    'loss': tiny_clip_loss.item(),
    'parameters': count_parameters(tiny_clip),
    'image_gradient_norm': tiny_clip.image_encoder.head.weight.grad.norm().item(),
    'text_gradient_norm': tiny_clip.text_projection.weight.grad.norm().item(),
}

## Day 17 practice and acceptance

1. Write the $8\times8$ similarity matrix and identify positives and negatives.
2. Compare image-to-text and text-to-image retrieval at 1.
3. Change temperature and observe loss and similarity concentration.
4. Construct a false-negative example with two valid captions for one visual concept.
5. List metadata or acquisition shortcuts that could create misleading retrieval performance.

**Accept the day when:** aligned pairs beat the permutation, the symmetric loss is explained, retrieval direction is explicit, and false negatives and shortcuts are acknowledged.

# Day 18 — Lightweight visual question answering and modality-use tests

The local VLM baseline mean-pools image patch tokens and non-padding question tokens, normalizes both vectors, concatenates them, and predicts a closed answer. This is inexpensive and interpretable, but successful training does not prove that both modalities are used.

The synthetic XOR task below requires both an image bit and a question bit. Majority, question-only, and image-only predictors are limited to 50%, while a fused nonlinear model can solve all four combinations. Controlled modality shuffles should damage the fused result.

In [ ]:
seed_everything(18)
image_bits = torch.tensor([0, 0, 1, 1], dtype=torch.long).repeat(8)
question_bits = torch.tensor([0, 1, 0, 1], dtype=torch.long).repeat(8)
vlm_labels = torch.bitwise_xor(image_bits, question_bits)
vlm_images = image_bits.float().view(-1, 1, 1, 1).expand(-1, 3, 32, 32).clone()
vlm_images += 0.02 * torch.randn_like(vlm_images)
question_token = question_bits + 1
vlm_questions = torch.stack(
    [question_token, torch.full_like(question_token, 3), torch.zeros_like(question_token)], dim=1
)

assert set(zip(image_bits.tolist(), question_bits.tolist(), vlm_labels.tolist())) == {
    (0, 0, 0), (0, 1, 1), (1, 0, 1), (1, 1, 0)
}
{
    'images': tuple(vlm_images.shape),
    'questions': tuple(vlm_questions.shape),
    'label_counts': dict(Counter(vlm_labels.tolist())),
}

In [ ]:
def lookup_baseline_accuracy(keys: torch.Tensor, labels: torch.Tensor) -> float:
    prediction_by_key = {}
    for key in keys.unique().tolist():
        candidates = labels[keys == key].tolist()
        prediction_by_key[key] = Counter(candidates).most_common(1)[0][0]
    predictions = torch.tensor([prediction_by_key[key] for key in keys.tolist()])
    return (predictions == labels).float().mean().item()

majority_label = Counter(vlm_labels.tolist()).most_common(1)[0][0]
majority_accuracy = (vlm_labels == majority_label).float().mean().item()
question_only_accuracy = lookup_baseline_accuracy(question_bits, vlm_labels)
image_only_accuracy = lookup_baseline_accuracy(image_bits, vlm_labels)
{
    'majority_accuracy': majority_accuracy,
    'question_only_accuracy': question_only_accuracy,
    'image_only_accuracy': image_only_accuracy,
}

In [ ]:
tiny_vlm = TinyVLMClassifier(vocab_size=8, n_answers=2, dim=16, padding_idx=0)
vlm_optimizer = torch.optim.Adam(tiny_vlm.parameters(), lr=0.02)
vlm_losses = []
tiny_vlm.train()
for step in range(120):
    vlm_logits = tiny_vlm(vlm_images, vlm_questions)
    vlm_loss = F.cross_entropy(vlm_logits, vlm_labels)
    vlm_optimizer.zero_grad(set_to_none=True)
    vlm_loss.backward()
    vlm_optimizer.step()
    vlm_losses.append(vlm_loss.item())

tiny_vlm.eval()
with torch.inference_mode():
    full_predictions = tiny_vlm(vlm_images, vlm_questions).argmax(dim=-1)
full_vlm_accuracy = (full_predictions == vlm_labels).float().mean().item()
assert full_vlm_accuracy >= 0.95
{'initial_loss': vlm_losses[0], 'final_loss': vlm_losses[-1], 'multimodal_accuracy': full_vlm_accuracy}

In [ ]:
image_permutation = torch.arange(len(vlm_images)).roll(2)
question_permutation = torch.arange(len(vlm_questions)).roll(1)
with torch.inference_mode():
    image_shuffled_predictions = tiny_vlm(
        vlm_images[image_permutation], vlm_questions
    ).argmax(dim=-1)
    question_shuffled_predictions = tiny_vlm(
        vlm_images, vlm_questions[question_permutation]
    ).argmax(dim=-1)
image_shuffled_accuracy = (image_shuffled_predictions == vlm_labels).float().mean().item()
question_shuffled_accuracy = (question_shuffled_predictions == vlm_labels).float().mean().item()

assert image_shuffled_accuracy < full_vlm_accuracy
assert question_shuffled_accuracy < full_vlm_accuracy
{
    'full_accuracy': full_vlm_accuracy,
    'image_shuffled_accuracy': image_shuffled_accuracy,
    'question_shuffled_accuracy': question_shuffled_accuracy,
}

In [ ]:
short_questions = vlm_questions[:4, :2]
padded_questions = vlm_questions[:4]
with torch.inference_mode():
    short_logits = tiny_vlm(vlm_images[:4], short_questions)
    padded_logits = tiny_vlm(vlm_images[:4], padded_questions)
torch.testing.assert_close(short_logits, padded_logits, atol=1e-6, rtol=1e-6)
{'padding_ignored': True, 'maximum_logit_difference': (short_logits - padded_logits).abs().max().item()}

## Day 18 practice and acceptance

1. Replace concatenation with elementwise product and compare capacity.
2. Train explicit question-only and image-only neural baselines with the same step budget.
3. Shuffle one modality at evaluation and explain what a performance drop does and does not prove.
4. Replace answer classification with a sketch of conditional text generation.
5. List which modules you would freeze, fully train, or adapt with LoRA in a pretrained system.

**Accept the day when:** the full model beats unimodal controls, both modality shuffles reduce performance, padding is ignored correctly, and architecture capacity is not confused with evidence of modality use.

# Day 19 — PathVQA protocol, data card, failures, and limitations

A real medical VQA experiment requires a data card containing provenance, license, image modality, question construction, de-identification, patient-level split policy, answer vocabulary, missing data, and known limitations. Duplicate images, patients, or templated questions across splits can invalidate results.

This notebook contains only four toy records. They are useful for metric plumbing, not scientific conclusions. Report yes/no and open questions separately, compare majority and question-only baselines, and review individual failures.

In [ ]:
toy_predictions = ['yes', 'circular region', 'no', 'purple']
metric_buckets = defaultdict(lambda: {'em': [], 'f1': []})
pathvqa_rows = []
for sample, prediction in zip(TOY_PATHVQA, toy_predictions):
    em = exact_match(prediction, sample['answer'])
    f1 = token_f1(prediction, sample['answer'])
    metric_buckets[sample['type']]['em'].append(em)
    metric_buckets[sample['type']]['f1'].append(f1)
    pathvqa_rows.append({**sample, 'prediction': prediction, 'em': em, 'token_f1': f1})

stratified_metrics = {
    question_type: {
        'exact_match': sum(values['em']) / len(values['em']),
        'token_f1': sum(values['f1']) / len(values['f1']),
        'n': len(values['em']),
    }
    for question_type, values in metric_buckets.items()
}
overall_majority = Counter(sample['answer'] for sample in TOY_PATHVQA).most_common(1)[0][0]
majority_accuracy = sum(sample['answer'] == overall_majority for sample in TOY_PATHVQA) / len(TOY_PATHVQA)
{'stratified_metrics': stratified_metrics, 'majority_answer': overall_majority, 'majority_accuracy': majority_accuracy}

In [ ]:
failure_records = []
for row in pathvqa_rows:
    if row['em'] == 1:
        continue
    if row['token_f1'] > 0:
        suspected_cause = 'partially correct wording or normalization mismatch'
    elif row['type'] == 'yes/no':
        suspected_cause = 'polarity error or visual evidence not used'
    else:
        suspected_cause = 'semantic or visual recognition error'
    failure_records.append({
        'question': row['question'],
        'target': row['answer'],
        'prediction': row['prediction'],
        'type': row['type'],
        'suspected_cause': suspected_cause,
    })
failure_records

## Day 19 practice and acceptance

1. Draft a real-data card before downloading any medical dataset.
2. Specify patient-, image-, and normalized-question leakage checks.
3. Report majority, question-only, image-only, and multimodal baselines by question type.
4. Expand the failure taxonomy to visual, linguistic, annotation, shortcut, and metric errors.
5. State that the model is educational, not clinically validated, and cannot replace expert review.

**Accept the day when:** provenance and split policy precede modeling, metrics are stratified, individual failures are recorded, and medical-use limitations are explicit.

# Day 20 — Capstone result table, reproduction package, and defense

The minimum capstone compares a majority baseline, question-only baseline, image-only baseline, multimodal model, and both modality-shuffle controls under one fixed protocol. The synthetic XOR experiment is a mechanism check, not a medical benchmark.

A reproduction package should include environment files, seeds, data-generation or split code, model configuration, training budget, metrics, result table, failure records, and exact commands. Claims must follow the evidence ladder: successful execution, tiny overfit, held-out evaluation, ablation evidence, and only then carefully bounded interpretation.

In [ ]:
capstone_results = [
    {'system': 'majority answer', 'uses_image': False, 'uses_question': False, 'accuracy': majority_accuracy},
    {'system': 'question-only lookup', 'uses_image': False, 'uses_question': True, 'accuracy': question_only_accuracy},
    {'system': 'image-only lookup', 'uses_image': True, 'uses_question': False, 'accuracy': image_only_accuracy},
    {'system': 'TinyVLM image + question', 'uses_image': True, 'uses_question': True, 'accuracy': full_vlm_accuracy},
    {'system': 'TinyVLM with shuffled images', 'uses_image': 'shuffled', 'uses_question': True, 'accuracy': image_shuffled_accuracy},
    {'system': 'TinyVLM with shuffled questions', 'uses_image': True, 'uses_question': 'shuffled', 'accuracy': question_shuffled_accuracy},
]
capstone_results

In [ ]:
reproduction_manifest = {
    'scope': 'synthetic mechanism check; no medical claims',
    'seed': 18,
    'data': '32 generated XOR image-question examples',
    'image_shape': list(vlm_images.shape[1:]),
    'model': 'TinyVLMClassifier',
    'model_parameters': count_parameters(tiny_vlm),
    'optimizer': 'Adam',
    'learning_rate': 0.02,
    'steps': 120,
    'metric': 'classification accuracy',
    'baselines': ['majority', 'question-only lookup', 'image-only lookup'],
    'ablations': ['image shuffle', 'question shuffle'],
    'environment': 'environment.yml / Python 3.11 / CPU PyTorch',
}
print(json.dumps(reproduction_manifest, indent=2))

# Final acceptance and ten-minute defense

- [ ] I can derive ViT token counts and explain CLS/position embeddings.
- [ ] I can write the symmetric CLIP objective and compute retrieval at 1.
- [ ] I can distinguish pooled fusion, cross-attention, and projector-to-LLM designs.
- [ ] My multimodal model beats majority and both unimodal controls.
- [ ] Image and question shuffles reduce performance in the controlled task.
- [ ] I report yes/no and open QA metrics separately.
- [ ] I audit source, license, de-identification, patient splits, and duplicates before real data use.
- [ ] I provide environment, seed, configuration, results, and failure records.
- [ ] I state that synthetic success does not imply real medical usefulness.

For the oral defense, explain the question, data protocol, architecture, baselines, training budget, results, shuffle evidence, failure cases, limitations, and one justified next experiment. The next experiment should improve evidence quality rather than merely increase model size.